In [ ]:
# install some necessary packages
!pip install wandb -qU
!pip install bitsandbytes -qU
!pip install peft -qU
!pip install datasets -qU
!pip install transformers -qU
!pip install loguru -qU
!pip install tokenizers -qU
!pip install sentencepiece -qU


In [ ]:
# connect to google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!huggingface-cli login

In [ ]:
# wandb login
import os
import wandb

# Set the WANDB_API_KEY environment variable
os.environ['WANDB_API_KEY'] = '87246aa35c0741ef13381fe3bead47e6d1eeebfb'  # Replace with your actual key

# Login to wandb
wandb.login()


In [ ]:
# import os and sys

OUTPUT_DIR = '/content/drive/MyDrive/Hallucination/Llama2_7b_Finance_FT_3'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

import sys

In [ ]:
from typing import List, Dict, Optional
from loguru import logger

import torch
import torch.nn as nn
import bitsandbytes as bnb
from datasets import load_dataset
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import (
    TaskType,
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    prepare_model_for_int8_training,
    set_peft_model_state_dict,
    get_peft_model_state_dict,
)

from peft.utils import TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING

In [ ]:
mmlu_dataset_management = load_dataset("cais/mmlu", name = 'management')

In [ ]:
mmlu_dataset_management['test']

In [ ]:
mmlu_dataset_marketing = load_dataset("cais/mmlu", name = 'marketing')

In [ ]:
type(mmlu_dataset_marketing)

# Set up training hyperparameters

In [ ]:
# prepare steps

# Set random seed for reproducibility
RANDOM_SEED = 3407
transformers.set_seed(RANDOM_SEED)

# Set up the training hyperparameter
MICRO_BATCH_SIZE = 64
BATCH_SIZE = 256
GRADIENT_ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE
EPOCHS = 1  # One epoch takes ~6 hours, and 2 epochs may exceed 12-hour.
LEARNING_RATE = 2e-5  # Following stanford_alpaca
CUTOFF_LEN = 256  # 256 accounts for about 96% of the data. Shorter input, faster training/less VRAM

LORA_R = 8  # Some LoRA parameters
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
VAL_SET_SIZE = 2000
TARGET_MODULES = [
    'q_proj',
    'v_prol',
]


In [ ]:
# DDP setting
device_map = 'auto'
world_size = int(os.environ.get('WORLD_SIZE', 1))
ddp = (world_size != 1)  # If more than one GPU, then DDP
if ddp:
    device_map = {'': int(os.environ.get('LOCAL_RANK') or 0)}
    GRADIENT_ACCUMULATION_STEPS = GRADIENT_ACCUMULATION_STEPS // world_size

# Helper functions

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
def generate_prompt(data_point):
    """Gen. input text based on a prompt, task instruction, (context info.), and answer

    :param data_point: dict: Data point
    :return: str: Input text
    """
    # Samples with additional context into.
    if data_point['input']:
        text = 'Below is an instruction that describes a task, paired with an input that provides' \
               ' further context. Write a response that appropriately completes the request.\n\n'
        text += f'### Instruction:\n{data_point["instruction"]}\n\n'
        text += f'### Input:\n{data_point["input"]}\n\n'
        text += f'### Response:\n{data_point["output"]}'
        return text

    # Without
    else:
        text = 'Below is an instruction that describes a task. Write a response that ' \
               'appropriately completes the request.\n\n'
        text += f'### Instruction:\n{data_point["instruction"]}\n\n'
        text += f'### Response:\n{data_point["output"]}'
        return text

In [ ]:
def tokenize(prompt):
    """Tokenize the input

    :param prompt: str: Input text
    :return: dict: {'tokenised input text': list, 'mask': list}
    """
    result = tokenizer(prompt, truncation=True, max_length=CUTOFF_LEN + 1, padding='max_length')
    return {
        'input_ids': result['input_ids'][:-1],
        'attention_mask': result['attention_mask'][:-1],
    }


In [ ]:
def generate_and_tokenize_prompt(data_point):
    """This function masks out the labels for the input, so that our loss is computed only on the
    response."""
    if data_point['input']:
        user_prompt = 'Below is an instruction that describes a task, paired with an input that ' \
                      'provides further context. Write a response that appropriately completes ' \
                      'the request.\n\n'
        user_prompt += f'### Instruction:\n{data_point["instruction"]}\n\n'
        user_prompt += f'### Input:\n{data_point["input"]}\n\n'
        user_prompt += f'### Response:\n'
    else:
        user_prompt = 'Below is an instruction that describes a task. Write a response that ' \
                      'appropriately completes the request.'
        user_prompt += f'### Instruction:\n{data_point["instruction"]}\n\n'
        user_prompt += f'### Response:\n'

    # Count the length of prompt tokens
    len_user_prompt_tokens = len(tokenizer(user_prompt,
                                           truncation=True,
                                           max_length=CUTOFF_LEN + 1,
                                           padding='max_length')['input_ids'])
    len_user_prompt_tokens -= 1  # Minus 1 (one) for eos token

    # Tokenise the input, both prompt and output
    full_tokens = tokenizer(
        user_prompt + data_point['output'],
        truncation=True,
        max_length=CUTOFF_LEN + 1,
        padding='max_length',
    )['input_ids'][:-1]
    return {
        'input_ids': full_tokens,
        'labels': [-100] * len_user_prompt_tokens + full_tokens[len_user_prompt_tokens:],
        'attention_mask': [1] * (len(full_tokens)),
    }


# Load Model&Tokenizer

In [ ]:
#from transformers import LlamaForCausalLM, LlamaTokenizer

model_name = "NousResearch/Llama-2-7b-hf"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    add_eos_token=True
)
#tokenizer.pad_token_id = 0 # unk. we want this to be different from the eos token

# Read Llama2-7b
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,
    trust_remote_code=True,
    device_map= 'auto',
)

model = prepare_model_for_int8_training(model, use_gradient_checkpointing=True)

# LoRA fine-tune

In [ ]:
# LoRA fine-tuning

# Set up LoRA config
lora_config = LoraConfig(
    task_type= 'CAUSAL_LM',
    r=LORA_R, # LORA_R
    lora_alpha= LORA_ALPHA, # LORA_ALPHA
    lora_dropout= LORA_DROPOUT, # LORA_DROPOUT
    target_modules= TARGET_MODULES, # TARGET_MODULES
    bias='none',
)
model = get_peft_model(model, lora_config)
print_trainable_parameters(model)


In [ ]:
# this snippet helps us retry and continue training on some checkpoints
resume_from_checkpoint = None
if resume_from_checkpoint is not None:
    checkpoint_name = os.path.join(resume_from_checkpoint, 'pytorch_model.bin')
    if not os.path.exists(checkpoint_name):
        checkpoint_name = os.path.join(
            resume_from_checkpoint, 'adapter_model.bin'
        )
        resume_from_checkpoint = False
    if os.path.exists(checkpoint_name):
        logger.info(f'Restarting from {checkpoint_name}')
        adapters_weights = torch.load(checkpoint_name)
        set_peft_model_state_dict(model, adapters_weights)
    else:
        logger.info(f'Checkpoint {checkpoint_name} not found')

In [ ]:
print_trainable_parameters(model)

# Load Datasets

In [ ]:
# Load data
data = load_dataset('gbharti/wealth-alpaca_lora')
data = data.shuffle(seed=RANDOM_SEED)  # Shuffle dataset here

In [ ]:
# Train/val split
if VAL_SET_SIZE > 0:
    train_val = data['train'].train_test_split(
        test_size=VAL_SET_SIZE, shuffle=False, seed=RANDOM_SEED
    )
    train_data = train_val['train'].map(generate_and_tokenize_prompt)
    val_data = train_val['test'].map(generate_and_tokenize_prompt)
else:
    train_data = data['train'].map(generate_and_tokenize_prompt)
    val_data = None

# Training

In [ ]:
# HuggingFace Trainer
# Fine-tune the model

trainer = transformers.Trainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    # trainer argument config
    args=transformers.TrainingArguments(
        seed=RANDOM_SEED,  # Reproducibility
        data_seed=RANDOM_SEED,
        per_device_train_batch_size=MICRO_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_steps=100,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=True,
        logging_steps=5,
        evaluation_strategy='steps' if VAL_SET_SIZE > 0 else 'no',
        save_strategy='steps',
        save_steps=10,
        eval_steps=5 if VAL_SET_SIZE > 0 else None,
        output_dir=OUTPUT_DIR,
        save_total_limit=3,
        load_best_model_at_end=True if VAL_SET_SIZE > 0 else False,
        ddp_find_unused_parameters=False if ddp else None,
    ),

    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
model.config.use_cache = False

# PEFT setup
old_state_dict = model.state_dict
model.state_dict = (
    lambda self, *_, **__: get_peft_model_state_dict(self, old_state_dict())
).__get__(model, type(model))

# Use the latest PyTorch 2.0 if possible
if torch.__version__ >= '2' and sys.platform != 'win32':
    model = torch.compile(model)

# Train
trainer.train()
wandb.finish()

In [ ]:
### Merge the LoRA adaptor to Base model

# Step 1: Extract LoRA Weights from the Trained Model
lora_weights = get_peft_model_state_dict(model)

# Step 2: Load the Original Llama2-7b Model
original_model = AutoModelForCausalLM.from_pretrained(model_name)

# Step 3: Update the Original Model with LoRA Weights
for key, value in lora_weights.items():
    if key in original_model.state_dict():
        original_model.state_dict()[key].data.copy_(value.data)

# Step 4: Save the Updated Model
# Specify your desired path to save the model
updated_model_path = os.path.join(OUTPUT_DIR, "Llama2-7b_Finance_FT_3_with_lora")
original_model.save_pretrained(updated_model_path)

print("LoRA parameters have been successfully merged into the original Llama2-7b model.")

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast   # 4.30.2
from peft import PeftModel  # 0.5.0
base_model = "NousResearch/Llama-2-13b-hf"
peft_model = "finetuned_model"
tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
model = LlamaForCausalLM.from_pretrained(base_model, trust_remote_code=True, device_map = "cuda:0", load_in_8bit = True,)
model = PeftModel.from_pretrained(model, peft_model)

# push to hub

In [ ]:
!git init "/content/drive/MyDrive/Hallucination/Llama2-7b_Finance_FT_3/git"
!git remote add origin YOUR_REPO_URL


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from huggingface_hub import HfApi, Repository

model_name = "Llama2-7b_Finance_lora_1"
model_path = OUTPUT_DIR
repo_path = "/content/drive/MyDrive/Hallucination/Llama2-7b_Finance_FT_1/git"
repo_name = "Llama2-7b_Finance_lora_1"  # Replace with your desired repository name

# Create a repository on the Hugging Face Hub
api = HfApi()
# Pass the repo_id argument with the name of the repository
repo_url = api.create_repo(token="HF_TOKEN_PLACEHOLDER", repo_id= repo_name, exist_ok=True)

# Clone the repository using the Hugging Face Hub library
repo = Repository(local_dir= repo_path, clone_from=repo_url)
repo.git_push()


In [ ]:
!rm -rf repo_path
!mkdir repo_path

In [ ]:

repo = Repository(local_dir=repo_path, clone_from="https://huggingface.co/Gason/Llama2-7b_Finance_lora_1")